In [ ]:
from tqdm import tqdm
import pandas as pd
import re

import os
import numpy as np
import sys
sys.path.append('../../mimic-cxr/txt')
import section_parser as sp

# Paths relative to this notebook (camchex/data/make_dataset.ipynb)
DATA_ROOT = '../../data'

mimic_cxr_metadata_fp  = f'{DATA_ROOT}/MIMIC-CXR-JPG/mimic-cxr-2.0.0-metadata.csv'
mimic_cxr_split_fp     = f'{DATA_ROOT}/MIMIC-CXR-JPG/mimic-cxr-2.0.0-split.csv'

mimic_iv_ed_triage_fp     = f'{DATA_ROOT}/MIMIC-IV-ED-2-2/mimic-iv-ed-2.2/ed/triage.csv.gz'
mimic_iv_ed_edstays_fp    = f'{DATA_ROOT}/MIMIC-IV-ED-2-2/mimic-iv-ed-2.2/ed/edstays.csv.gz'
mimic_iv_ed_vitalsigns_fp = f'{DATA_ROOT}/MIMIC-IV-ED-2-2/mimic-iv-ed-2.2/ed/vitalsign.csv.gz'

output_fp = f'{DATA_ROOT}/dataset.csv'

# CXR-LT 2023 labels files
_CXRLT_2023 = f'{DATA_ROOT}/CXR-LT/cxr-lt-multi-label-long-tailed-classification-on-chest-x-rays-2.0.0/cxr-lt-2023'
labels_train_fp    = f'{_CXRLT_2023}/train.csv'
labels_validate_fp = f'{_CXRLT_2023}/development.csv'
labels_test_fp     = f'{_CXRLT_2023}/test.csv'

# Adapted from https://github.com/zhjohnchan/R2Gen/blob/main/modules/tokenizers.py
def _clean_report(report):
    report = report.replace('\\n', ' ').replace('\n', ' ')
    
    report = re.sub(r'(?<![a-zA-Z])/{1,}|/{1,}(?![a-zA-Z])', '', report)
    
    report = re.sub(r'\s+', ' ', report).strip()

    report_cleaner = lambda t: t.replace('__', '_') \
        .replace('..', '.').replace('  ', ' ') \
        .replace('1. ', '').replace('2. ', '').replace('3. ', '') \
        .replace('4. ', '').replace('5. ', '') \
        .strip().lower()

    sent_cleaner = lambda t: re.sub(r'[.,?;*!%^&_+():-\[\]{}]', '', t.replace('"', '')
                                    .replace("'", '')
                                    .strip().lower())

    cleaned_sentences = [sent_cleaner(sent) for sent in report_cleaner(report).split('. ') if sent]
    cleaned_report = ' . '.join(cleaned_sentences).strip() + ' .'
    
    cleaned_report = re.sub(r'\s+', ' ', cleaned_report).strip()
    return cleaned_report

In [ ]:

metadata_df = pd.read_csv(mimic_cxr_metadata_fp)
cxr_df = metadata_df.drop(columns=['dicom_id', 'ViewPosition', 'Rows', 'Columns']).drop_duplicates(subset='study_id').reset_index(drop=True)

In [ ]:
cxr_df = cxr_df.sort_values(by=['subject_id', 'StudyDate']).reset_index(drop=True)
cxr_df['PreviousStudy'] = cxr_df.groupby('subject_id')['study_id'].shift(1).astype('Int64')

cxr_df['StudyDateTime'] = (cxr_df['StudyDate'] * 1000000) + cxr_df['StudyTime'].astype(int)
cxr_df['StudyDateTime'] = pd.to_datetime(cxr_df['StudyDateTime'], format="%Y%m%d%H%M%S")

cxr_df = cxr_df.drop(columns=['StudyDate', 'StudyTime'])

triage_df = pd.read_csv(mimic_iv_ed_triage_fp)
edstays_df = pd.read_csv(mimic_iv_ed_edstays_fp)

ed_df = pd.merge(triage_df, edstays_df, on=['subject_id', 'stay_id'])

ed_df['intime'] = pd.to_datetime(ed_df['intime'])
ed_df['outtime'] = pd.to_datetime(ed_df['outtime'])
cxr_df['StudyDateTime'] = pd.to_datetime(cxr_df['StudyDateTime'])

mimic_df = pd.merge(
    cxr_df,
    ed_df,
    on='subject_id',
    how='left'
)

mimic_df['time_to_intime'] = (mimic_df['StudyDateTime'] - mimic_df['intime']).abs().dt.total_seconds()
mimic_df['time_to_outtime'] = (mimic_df['StudyDateTime'] - mimic_df['outtime']).abs().dt.total_seconds()

closest_stays_idx = mimic_df.dropna(subset=['time_to_intime']).groupby('study_id')['time_to_intime'].idxmin()

mimic_df = pd.concat([
    mimic_df.loc[closest_stays_idx],
    mimic_df[~mimic_df['study_id'].isin(mimic_df.loc[closest_stays_idx, 'study_id'])]
])

mimic_df = mimic_df.drop(columns=['time_to_intime', 'time_to_outtime']).reset_index(drop=True)

vitalsigns_df = pd.read_csv(mimic_iv_ed_vitalsigns_fp)

mimic_df['StudyDateTime'] = pd.to_datetime(mimic_df['StudyDateTime'])
vitalsigns_df['charttime'] = pd.to_datetime(vitalsigns_df['charttime'])

merged_df = pd.merge(
    mimic_df, 
    vitalsigns_df, 
    on=['subject_id', 'stay_id'], 
    suffixes=('', '_chart'), 
    how='left'
)

merged_df['time_diff'] = (merged_df['StudyDateTime'] - merged_df['charttime']).abs()

vital_signs = ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain']  


closest_charts = (
    merged_df[merged_df[vital_signs].isna().any(axis=1)]
    .sort_values(by=['study_id', 'time_diff']) 
    .drop_duplicates(subset=['study_id'], keep='first')  
)

for vital in vital_signs:
    mimic_df[vital] = mimic_df[vital].combine_first(closest_charts[f"{vital}_chart"])

mimic_df = mimic_df.reset_index(drop=True)

label_cols = [
    "study_id", "Atelectasis", "Calcification of the Aorta", "Cardiomegaly", "Consolidation", "Edema", "Emphysema",
    "Enlarged Cardiomediastinum", "Fibrosis", "Fracture", "Hernia", "Infiltration", "Lung Lesion", "Lung Opacity",
    "Mass", "No Finding", "Nodule", "Pleural Effusion", "Pleural Other", "Pleural Thickening", "Pneumomediastinum",
    "Pneumonia", "Pneumoperitoneum", "Pneumothorax", "Subcutaneous Emphysema", "Support Devices", "Tortuous Aorta"
]

df_train = pd.read_csv(labels_train_fp, usecols=label_cols)
df_val = pd.read_csv(labels_validate_fp, usecols=label_cols)
df_test = pd.read_csv(labels_test_fp, usecols=label_cols)

df_train['split'] = 'train'
df_val['split'] = 'validate'
df_test['split'] = 'test'

labels_df = pd.concat([df_train, df_val, df_test]).drop_duplicates(subset="study_id")

cxr_df = cxr_df.merge(labels_df, on="study_id", how="left")


In [ ]:
reports_base_path = f"{DATA_ROOT}/MIMIC-CXR/files"

custom_section_names, custom_indices = sp.custom_mimic_cxr_rules()

mimic_df['impression'] = None
mimic_df['findings'] = None
mimic_df['last_paragraph'] = None
mimic_df['comparison'] = None
mimic_df['indication'] = None  
mimic_df['history'] = None

for i, row in tqdm(mimic_df.iterrows(), total=len(mimic_df), desc="Processing reports"):
    subject_id_str = str(row['subject_id'])
    study_id_str = f"s{row['study_id']}"
    p_folder = f"p{subject_id_str[:2]}" 
    patient_folder = f"p{subject_id_str}"
    study_file = f"{study_id_str}.txt"
    report_path = os.path.join(reports_base_path, p_folder, patient_folder, study_file)

    if not os.path.exists(report_path):
        print(f"Report not found: {report_path}")
        continue  

    with open(report_path, 'r') as file:
        text = file.read()

    if study_id_str in custom_indices:
        idx = custom_indices[study_id_str]
        text = text[idx[0]:idx[1]]
    
    sections, section_names, _ = sp.section_text(text)

    section_dict = dict(zip(section_names, sections))

    if study_id_str in custom_section_names:
        custom_section = custom_section_names[study_id_str]
        if custom_section in section_dict:
            mimic_df.at[i, custom_section] = _clean_report(section_dict[custom_section])
        continue  

    for section in ['impression', 'findings', 'last_paragraph', 'comparison', 'indication', 'history']:
        if section in section_dict:
            mimic_df.at[i, section] = _clean_report(section_dict[section])


In [ ]:
mimic_df.to_csv(f'{DATA_ROOT}/progress.csv', index=False)

In [ ]:
def choose_report(row):
    if pd.notna(row['findings']) and str(row['findings']).strip():
        return row['findings']
    elif pd.notna(row['impression']) and str(row['impression']).strip():
        return row['impression']
    elif pd.notna(row['last_paragraph']) and str(row['last_paragraph']).strip():
        return row['last_paragraph']
    else:
        return np.nan

mimic_df['report'] = mimic_df.apply(choose_report, axis=1)

for study_id, section_name in custom_section_names.items():
    sid = int(study_id[1:])  
    if section_name in mimic_df.columns:
        custom_value = mimic_df.loc[mimic_df['study_id'] == sid, section_name]
        mimic_df.loc[mimic_df['study_id'] == sid, 'report'] = custom_value
        mimic_df.loc[mimic_df['study_id'] == sid, section_name] = np.nan

def merge_clinical_indication(row):
    ind = str(row['indication']) if pd.notna(row['indication']) else ''
    hist = str(row['history']) if pd.notna(row['history']) else ''
    combined = ind.strip()
    if hist.strip():
        combined += (' ' if combined else '') + hist.strip()
    return combined if combined else np.nan

mimic_df['ClinicalIndication'] = mimic_df.apply(merge_clinical_indication, axis=1)

cols_to_drop = [
    'impression', 'findings', 'last_paragraph', 'comparison', 'indication',
    'examination', 'technique', 'recommendations', 'history', 'note'
]
mimic_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [ ]:
mimic_df.columns

In [ ]:
meta_df = metadata_df[['dicom_id', 'study_id', 'ViewPosition']]

In [ ]:
mimic_df.drop(columns=['PerformedProcedureStepDescription', 'ProcedureCodeSequence_CodeMeaning', 'ViewCodeSequence_CodeMeaning', 'PatientOrientationCodeSequence_CodeMeaning'], inplace=True)

In [ ]:
merged_df = mimic_df.merge(
    meta_df[['study_id', 'dicom_id', 'ViewPosition']],
    on='study_id',
    how='left'
)

In [ ]:
mimic_df = merged_df.dropna(subset=['report'])

In [ ]:
merged_df = merged_df.rename(columns={'ClinicalIndication':'clinical_indication'})

In [ ]:
merged_df.to_csv(output_fp, index=False)

In [ ]:
labels_train_df = pd.read_csv(labels_train_fp)
labels_validate_df = pd.read_csv(labels_validate_fp)
labels_test_df = pd.read_csv(labels_test_fp)

labels_df = pd.concat([labels_train_df, labels_validate_df, labels_test_df])


In [ ]:
labels_df.drop(columns=['subject_id', 'study_id', 'ViewPosition',
       'ViewCodeSequence_CodeMeaning', 'path',], inplace=True)

In [ ]:
merged_df = merged_df.merge(labels_df, on='dicom_id', how='left')

In [ ]:
merged_df.columns

In [ ]:
merged_df.to_csv(f'{DATA_ROOT}/dataset.csv', index=False)

In [ ]:
merged_df['path'] = merged_df.apply(
    lambda row: f"images/p{str(row['subject_id'])[:2]}/p{str(row['subject_id'])}/s{row['study_id']}/{row['dicom_id']}.jpg",
    axis=1
)

In [ ]:
train = merged_df[merged_df['split'] == 'train']
validate = merged_df[merged_df['split'] == 'validate']
test = merged_df[merged_df['split'] == 'test']

In [ ]:
train = train.drop(columns=['split'])
validate = validate.drop(columns=['split'])
test = test.drop(columns=['split'])

In [ ]:
train.to_csv(f'{DATA_ROOT}/train.csv', index=False)
validate.to_csv(f'{DATA_ROOT}/development.csv', index=False)
test.to_csv(f'{DATA_ROOT}/test.csv', index=False)